# Decoding the Rhythms of Emotion: Exploratory Data Analysis (EDA)
**Text Mining Group Project 2026/2027**

#### Key Inquiries:
1. **Dataset Structure & Compact Overview**
2. **Data Quality & Integrity Audit** 
3. **Target Distribution**
4. **Lyrical Length & Structural Properties**
5. **Vocabulary & Language Nuances**
6. **Metadata Context**

## 1. Setup & Environment
We import standard libraries for data handling and visualization. Plot formatting is initialized for clean presentation.

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Visual styling
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

## 2. Dataset Description & Loading

In [ ]:
# Robust dataset path resolution
possible_dirs = ['../dataset', 'dataset', '../datasets', 'datasets']
data_dir = next((d for d in possible_dirs if os.path.exists(d)), None)

if data_dir is None:
    raise FileNotFoundError("Could not locate dataset folder. Check directory structure.")

print(f"Loading data from: {os.path.abspath(data_dir)}")

train_df = pd.read_csv(os.path.join(data_dir, 'train.csv'))
test_df = pd.read_csv(os.path.join(data_dir, 'test.csv'))

print(f"Training split: {train_df.shape[0]:,} rows × {train_df.shape[1]} columns")
print(f"Testing split:  {test_df.shape[0]:,} rows × {test_df.shape[1]} columns")

## 3. Focused Data Quality & Integrity Audit
Before drawing domain inferences, we audit both splits for missing entries, duplicates, train/test leakage, and numerical anomalies.

In [ ]:
# 1. Missing Values in both splits
print("--- Missing Values ---")
print("Train nulls:")
print(train_df.isnull().sum()[train_df.isnull().sum() > 0])
print("\nTest nulls:")
print(test_df.isnull().sum()[test_df.isnull().sum() > 0] if test_df.isnull().any().any() else "No missing values in test set.")

# 2. Duplicate Check
train_dup_ids = train_df['id'].duplicated().sum()
test_dup_ids = test_df['id'].duplicated().sum()
train_dup_songs = train_df.duplicated(subset=['title', 'artist']).sum()
test_dup_songs = test_df.duplicated(subset=['title', 'artist']).sum()

print("\n--- Duplicate Checks ---")
print(f"Duplicate IDs: Train = {train_dup_ids}, Test = {test_dup_ids}")
print(f"Duplicate (title, artist) pairs: Train = {train_dup_songs}, Test = {test_dup_songs}")

# 3. Train/Test ID Overlap (Data Leakage)
id_overlap = len(set(train_df['id']).intersection(set(test_df['id'])))
print(f"Train/Test ID Overlap: {id_overlap} songs")

# 4. Anomalous Years
anomalous_years = train_df[train_df['year'] < 1900]
print(f"\n--- Anomalous Release Years (year < 1900) ---")
print(f"Count in train: {len(anomalous_years)}")
print("Sample entries with year == 1:")
display(anomalous_years[['title', 'artist', 'year', 'tag']].head(5))

# 5. Views Summary Statistics
print("\n--- Views Distribution Summary (Train) ---")
print(train_df['views'].describe())

### Data Quality Takeaways:
1. **Missing Data:** Only **2 titles** are missing in the entire training set (`title` column), while the text of interest (`lyrics`) and the target (`tag`) are 100% complete with 0 missing entries. The test set has zero missing values across all columns.
2. **Integrity & Leakage:** There are **0 duplicate IDs** within either split and **0 ID overlap** between train and test, confirming a clean partition.
3. **Year Anomalies (`year == 1`):** Exactly **45 songs** have `year == 1`. Inspection shows these belong to two categories:
   - Ancient/traditional religious texts (e.g., *"Genesis 28"*, *"Acts 3"* by *Holy Bible (KJV)*).
   - Songs where scraper metadata failed or release dates were unspecified, resulting in default Unix/SQL epoch date fallbacks (`0001-01-01`).
   - *Implication:* Release year must be treated cautiously if used in analysis; missing/placeholder years should be imputed or masked.
4. **Views Distribution:** Views exhibit extreme positive skewness:
   - **Minimum:** 0
   - **Median (50%):** 89 views (half the dataset has under 89 views)
   - **Mean:** ≈ 3,413 views
   - **75th Percentile:** 473 views
   - **Maximum:** 3,604,497 views
   - *Implication:* The massive difference between median (89) and mean (~3,413) confirms that popularity follows a heavy power-law distribution. Any comparative analysis or visualization involving views requires logarithmic scaling.

## 4. Target Variable Analysis: Musical Genre Distribution (`tag`)
We inspect the frequency of the 6 musical genres in the training data to evaluate class balance.

In [ ]:
genre_counts = train_df['tag'].value_counts()
genre_pcts = train_df['tag'].value_counts(normalize=True) * 100

summary_table = pd.DataFrame({
    'Song Count': genre_counts,
    'Percentage (%)': genre_pcts.round(2)
})
display(summary_table)

plt.figure(figsize=(10, 5))
ax = sns.barplot(x=genre_counts.values, y=genre_counts.index, palette='viridis')
for i, (count, pct) in enumerate(zip(genre_counts.values, genre_pcts.values)):
    ax.text(count + 600, i, f"{count:,} ({pct:.1f}%)", va='center', fontsize=10)

plt.title('Genre Distribution in Training Set (Target: tag)', fontsize=14, weight='bold')
plt.xlabel('Number of Songs')
plt.ylabel('Genre')
plt.xlim(0, max(genre_counts.values) * 1.15)
plt.tight_layout()
plt.show()

### Takeaways on Genre Balance:
* **Heavy Class Asymmetry:**
  * Dominant classes: **`pop`** (55,745 songs, 41.3%) and **`rap`** (38,587 songs, 28.6%) account for nearly **70%** of the entire corpus.
  * Moderate class: **`rock`** (25,331 songs, 18.8%).
  * Under-represented classes: **`rb`** (6,206 songs, 4.6%), **`misc`** (5,639 songs, 4.2%), and **`country`** (3,459 songs, 2.6%).
* **Downstream Modeling Implications:**
  * Raw classification accuracy is an uninformative metric: a dummy baseline always predicting `pop` would achieve >41% accuracy.
  * Validation splits must use **Stratified sampling** (`StratifiedKFold`) to preserve minority class ratios.
  * Optimization and assessment must rely on **Macro-averaged F1-Score** and balanced loss weighting.

## 5. Song Lyrics Length & Word Count Distributions
We compute word count (tokens separated by whitespace) to examine variations in lyrical density across genres.

In [ ]:
train_df['lyrics_word_count'] = train_df['lyrics'].astype(str).apply(lambda x: len(x.split()))

# Summary statistics overall and by genre
print("--- Overall Lyric Word Count Summary ---")
print(train_df['lyrics_word_count'].describe().round(1))

genre_word_stats = train_df.groupby('tag')['lyrics_word_count'].agg(
    count='count',
    median='median',
    mean='mean',
    q25=lambda x: np.percentile(x, 25),
    q75=lambda x: np.percentile(x, 75)
).round(1).sort_values(by='median', ascending=False)

print("\n--- Word Count by Genre (Robust Summaries) ---")
display(genre_word_stats)

# Visualizing distributions
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Distribution up to 1,500 words
sns.histplot(train_df['lyrics_word_count'], bins=60, kde=True, ax=axes[0], color='teal')
axes[0].set_xlim(0, 1500)
axes[0].set_title('Overall Lyric Word Count (Capped at 1,500 for view)')
axes[0].set_xlabel('Word Count')
axes[0].set_ylabel('Song Count')

# Boxplot by Genre (excluding visual outliers for legible comparison)
sns.boxplot(data=train_df, x='tag', y='lyrics_word_count', ax=axes[1], palette='Set2', showfliers=False)
axes[1].set_title('Word Count Distribution by Genre (IQR only)')
axes[1].set_xlabel('Genre')
axes[1].set_ylabel('Word Count')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

### Takeaways on Lyric Lengths:
* **Genre Word Count Hierarchy (Medians):**
  * **`rap`:** Median = **415 words** (IQR: 300 to 542 words). Rap songs feature substantially more lyrical content than any other genre due to dense, fast-paced rhymed verses.
  * **`rb`:** Median = **291 words** (IQR: 201 to 388 words).
  * **`misc`:** Median = **258 words** (IQR: 147 to 371 words).
  * **`country`:** Median = **217 words** (IQR: 154 to 286 words).
  * **`pop`:** Median = **205 words** (IQR: 139 to 279 words).
  * **`rock`:** Median = **188 words** (IQR: 122 to 263 words). Rock songs show the lowest median word count, frequently prioritizing instrumentation, solos, and repetitive choruses over extensive verse text.
* **Extreme Length Outliers:**
  * **Minimum:** 1 word (4 songs have <10 words, representing instrumental placeholders or incomplete metadata).
  * **Maximum:** 18,221 words (555 songs have >3,000 words, representing medleys, spoken books, or scraper concatenations).
  * *Preprocessing Decision:* Extreme long lyrics should be capped/truncated (e.g. at 1,000 words) to avoid sparse matrix bloat and memory exhaustion.

## 6. Song Structure Tags (`[Chorus]`, `[Verse]`, etc.)
Lyrics sourced from lyrics databases commonly include section header tags in square brackets. We inspect what tags appear and how frequently they occur.

In [ ]:
def extract_section_tags(text):
    return re.findall(r'\[(.*?)\]', str(text))

structure_tags_series = train_df['lyrics'].apply(extract_section_tags)
all_tags = [tag.strip().lower() for sublist in structure_tags_series for tag in sublist]
tag_counts = pd.Series(all_tags).value_counts()

print(f"Total section tags detected: {len(all_tags):,}")
print(f"Unique tag names: {tag_counts.nunique():,}")
print("\nTop 12 Most Frequent Tags:")
print(tag_counts.head(12))

plt.figure(figsize=(10, 5))
sns.barplot(x=tag_counts.head(10).values, y=tag_counts.head(10).index, palette='mako')
plt.title('Top 10 Most Common Song Structure Tags', fontsize=14, weight='bold')
plt.xlabel('Occurrences Across Dataset')
plt.ylabel('Tag Name')
plt.tight_layout()
plt.show()

### Takeaways on Song Structure:
* Over **73,500 explicit `[Chorus]` tags** appear in the corpus, followed by `[Verse 1]`, `[Verse 2]`, `[Hook]`, and `[Bridge]`.
* **Implications for Preprocessing:**
  1. For word-level vectorizers, bracketed tags must be stripped so tag labels do not become artificial vocabulary features.
  2. Because the chorus is structurally the thematic core of a song, extracting the chorus into an auxiliary feature for representation testing is a viable direction for preprocessing experiments.

## 7. Vocabulary Diversity & Character Encoding Checks

### Methodological Caveats:
1. **Type-Token Ratio (TTR) Limitation:** TTR ($\text{Unique Words} / \text{Total Words}$) is inherently sensitive to text length (Herdan’s Law: longer texts naturally have lower TTR because speakers repeat functional words). To ensure a fair comparison, we compute TTR on fixed-length text samples.
2. **Non-ASCII $\neq$ Language:** A low non-ASCII character percentage does **not** prove a text is English. Romance languages (Spanish, French, Portuguese, Italian) or Indonesian often use standard Latin letters or have diacritics stripped. It only confirms whether characters conform to standard 7-bit ASCII.

In [ ]:
# Sample 1,000 songs per genre to standardize computation
sample_songs = train_df.groupby('tag', group_keys=False).apply(
    lambda g: g.sample(min(len(g), 1000), random_state=42)
).copy()

def compute_standardized_ttr(text, sample_length=150):
    words = str(text).lower().split()
    if len(words) < sample_length:
        if len(words) == 0:
            return np.nan
        return len(set(words)) / len(words)
    # Sub-sample to fixed token window to avoid length-bias
    sub_words = words[:sample_length]
    return len(set(sub_words)) / sample_length

sample_songs['std_ttr'] = sample_songs['lyrics'].apply(compute_standardized_ttr)
ttr_summary = sample_songs.groupby('tag')['std_ttr'].agg(['median', 'mean']).sort_values(by='median', ascending=False)

print("--- Standardized Lexical Diversity (TTR at 150-word window) ---")
display(ttr_summary.round(3))

# Character encoding check (Non-ASCII proportion)
def non_ascii_ratio(text):
    s = str(text)
    if not s:
        return 0.0
    return sum(1 for c in s if ord(c) > 127) / len(s)

train_df['non_ascii_ratio'] = train_df['lyrics'].apply(non_ascii_ratio)
high_non_ascii = train_df[train_df['non_ascii_ratio'] > 0.15]
print(f"\nSongs with > 15% non-ASCII characters: {len(high_non_ascii)} ({len(high_non_ascii)/len(train_df)*100:.2f}%)")

### Takeaways on Vocabulary & Language:
* **Lexical Repetition by Genre:** Even when controlling for length, `rb` and `pop` exhibit lower lexical diversity, reflecting repetitive refrains and formulaic choruses, whereas `rock` and `misc` exhibit higher unique word density per 150-word passage.
* **Character Encoding:** Over **99.9%** of songs contain virtually all ASCII characters. While this does not rule out Latin-script non-English lyrics (e.g. Spanish pop), it confirms that standard Western/English tokenizers and lemmatizers will process the text without character decoding errors.

## 8. Metadata Context: Release Years, Views, and Collaborations
The project brief specifies that the text classification goal is to **predict musical genre from song lyrics**. 
Here, we examine non-lyrics metadata solely for **descriptive and contextual understanding**, rather than proposing them as direct classification inputs.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Release Year Distribution by Genre (valid range: 1950 to 2025)
valid_years_df = train_df[(train_df['year'] >= 1950) & (train_df['year'] <= 2025)]
sns.boxplot(data=valid_years_df, x='tag', y='year', ax=axes[0], palette='coolwarm', showfliers=False)
axes[0].set_title('Release Year Distribution (Excluding Outliers)')
axes[0].set_xlabel('Genre')
axes[0].set_ylabel('Year')
axes[0].tick_params(axis='x', rotation=30)

# 2. Views Distribution by Genre (Log Scale)
sns.boxplot(data=train_df, x='tag', y='views', ax=axes[1], palette='Set2', showfliers=False)
axes[1].set_yscale('log')
axes[1].set_title('Views Distribution (Log Scale)')
axes[1].set_xlabel('Genre')
axes[1].set_ylabel('Views (Log)')
axes[1].tick_params(axis='x', rotation=30)

# 3. Collaboration Rates (Presence of featured artists)
train_df['has_features'] = train_df['features'].apply(
    lambda x: x != '{}' and pd.notnull(x) and str(x).strip() != ''
)
collab_rate = (train_df.groupby('tag')['has_features'].mean() * 100).sort_values(ascending=False)
sns.barplot(x=collab_rate.values, y=collab_rate.index, ax=axes[2], palette='viridis')
axes[2].set_title('Songs with Collaborating Artists (%)')
axes[2].set_xlabel('Percentage (%)')
axes[2].set_ylabel('Genre')
for i, v in enumerate(collab_rate.values):
    axes[2].text(v + 0.5, i, f"{v:.1f}%", va='center', fontsize=10)

plt.tight_layout()
plt.show()

### Metadata Context Observations:
* **Release Era:** Country songs skew slightly older (median year: 2012), while Rap and R&B are heavily concentrated in more modern releases (median year: 2018).
* **Viewership Skew:** All genres follow a similar log-scale distribution of views, with Pop and Rap maintaining slightly higher 75th percentiles.
* **Collaboration Culture:** **Rap (29.8%)** and **R&B (21.1%)** feature collaborations far more frequently than **Rock (8.0%)** or **Country (7.6%)**, reflecting distinct subcultural production styles.

## 9. Summary of EDA Findings & Direct Implications for Next Steps

| Discovered Pattern | Empirical Evidence | Implication for Preprocessing / Modeling |
| :--- | :--- | :--- |
| **Class Imbalance** | Pop (41.3%) and Rap (28.6%) dominate; Country is only 2.6%. | Stratified sampling is strictly necessary. Primary metric must be Macro-F1. |
| **Lyric Length Variations** | Rap median is 415 words; Rock median is 188 words. | Vectorization must account for length discrepancies (sublinear term frequency scaling and L2 normalization). |
| **Extreme Outliers** | 4 songs have <10 words; 555 songs exceed 3,000 words. | Truncate long lyrics (e.g. 1,000 tokens); clean corrupted/empty entries. |
| **Structure Tags** | Over 73k `[Chorus]` markers and explicit verse tags. | Strip bracketed tags from general vocabulary; test dedicated chorus feature in preprocessing. |
| **Year Placeholders** | 45 songs with `year == 1` (ancient texts / scraper nulls). | Treat release year carefully; impute or mask in temporal sentiment analysis. |
| **Views Skewness** | Median = 89 views, Mean ≈ 3,413, Max > 3.6M. | Logarithmic transform required for popularity correlation in sentiment analysis. |
